# 2 · Agentes de IA — Ciclo ReAct manual (Gemini)
### Sesión 1 — Agentes de IA, Orquestación y Protocolos

**Complejidad: 🟢 Básica**   |   **Dependencias: solo `google-genai`**

Construimos, sin ningún framework, el loop **razonar → actuar → observar** que vimos en el diagrama de la charla. La diferencia con el notebook 1 es que aquí el ciclo se repite **N veces**, encadenando interacciones con `previous_interaction_id`, hasta que el modelo decide que ya tiene todo lo necesario.

Este notebook es completamente autocontenido (no depende de haber corrido el notebook 1).


## 🎯 Objetivo de aprendizaje

Al terminar este notebook vas a poder:
- Explicar la diferencia entre un chatbot y un **agente de IA**.
- Describir el ciclo ReAct (razonar → actuar → observar) y por qué es la base de cualquier agente.
- Implementar ese ciclo a mano, sin frameworks, controlando explícitamente cuándo se detiene (respuesta final vs. límite de iteraciones).


## 📚 Teoría: Agentes de IA y el ciclo ReAct

Un **agente de IA** no es solo un modelo que responde una pregunta — es un modelo que puede **decidir una secuencia de acciones** para lograr un objetivo, usando herramientas y observando los resultados de sus propias acciones antes de decidir el siguiente paso.

El patrón que sostiene esto se llama **ReAct** (*Reason + Act*): el modelo alterna entre razonar ("¿qué necesito hacer ahora?") y actuar (invocar una herramienta), usando lo que observa en cada resultado para decidir si ya puede responder o si necesita otro paso. Es el mismo ciclo de function calling del notebook 1, pero repetido **N veces** en vez de una sola vuelta.

El ciclo termina en alguno de estos tres casos:
- El modelo decide que ya tiene toda la información necesaria y da la respuesta final.
- Se alcanza un límite explícito de iteraciones (`max_steps`) — un control de seguridad imprescindible para evitar loops infinitos.
- Ocurre un error que requiere intervención humana.

**Niveles de autonomía:** un agente puede diseñarse en un espectro que va desde *asistido* (una persona ejecuta cada acción manualmente, el modelo solo sugiere) hasta *autónomo* (decide y ejecuta sin supervisión, con monitoreo posterior). A mayor autonomía, mayor productividad — pero también mayor riesgo, por lo que la elección del nivel correcto depende del caso de uso.


## 0. Instalación (única dependencia)

In [ ]:
!pip install -q google-genai

### Configurar API key de Gemini

**Cómo obtenerla:** [aistudio.google.com/apikey](https://aistudio.google.com/apikey) (gratis, dos clics).

Recomendado en Colab: guárdala en **Secrets** (ícono de llave 🔑 a la izquierda) con el nombre `GEMINI_API_KEY` y actívala para este notebook. Si no usas Secrets, te la pedirá por input.

⚠️ **Aviso conocido (2026):** Google está migrando las API keys al nuevo formato con prefijo `AQ.` (antes `AIza...`). Hay reportes activos y aún no resueltos en el foro oficial de Google de que las keys `AQ.` devuelven `401 ACCESS_TOKEN_TYPE_UNSUPPORTED` en algunas cuentas/proyectos, incluso bien configuradas. La celda de abajo te dice qué tipo de key tienes para descartar esto como causa del error.


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["GEMINI_API_KEY"] = os.environ.get("GEMINI_API_KEY") or getpass("Pega tu GEMINI_API_KEY: ")

_key = os.environ.get("GEMINI_API_KEY", "")
print("API key configurada:", "OK" if _key else "FALTA")

if _key.startswith("AQ."):
    print("ADVERTENCIA: tu key tiene el nuevo formato 'AQ.' -- si mas adelante ves un error 401")
    print("ACCESS_TOKEN_TYPE_UNSUPPORTED, es un problema conocido y actualmente activo del lado de")
    print("Google con este formato de key, no de este notebook. Revisa:")
    print("https://discuss.ai.google.dev/c/gemini-api/4  (buscar 'AQ. 401 ACCESS_TOKEN_TYPE_UNSUPPORTED')")
elif _key.startswith("AIza"):
    print("Formato de key clasico (AIza...) -- no deberia verse afectado por el problema de las keys 'AQ.'.")


## 1. Herramientas del agente

In [ ]:
from google import genai

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])  # explícito: evita depender de la autodetección de entorno
MODEL = "gemini-3.5-flash"

def calculadora(expresion: str) -> str:
    """Evalúa una expresión matemática simple."""
    try:
        return str(eval(expresion, {"__builtins__": {}}))
    except Exception as e:
        return f"Error: {e}"

def buscar_evento_universidad(tema: str) -> str:
    """Simula una búsqueda en el calendario de eventos de la universidad."""
    eventos = {
        "ia": "Seminario de Inteligencia Artificial - 25 de julio, Auditorio Principal, 3pm",
        "emprendimiento": "Feria de Emprendimiento - 30 de julio, Plazoleta Central, 9am",
    }
    for k, v in eventos.items():
        if k in tema.lower():
            return v
    return "No se encontraron eventos relacionados con ese tema."

agent_tools = [
    {
        "type": "function",
        "name": "calculadora",
        "description": "Evalúa expresiones matemáticas. Úsala para cualquier cálculo numérico.",
        "parameters": {"type": "object", "properties": {"expresion": {"type": "string"}}, "required": ["expresion"]}
    },
    {
        "type": "function",
        "name": "buscar_evento_universidad",
        "description": "Busca eventos en el calendario de la universidad por tema.",
        "parameters": {"type": "object", "properties": {"tema": {"type": "string"}}, "required": ["tema"]}
    },
]

tool_implementations = {
    "calculadora": lambda expresion: calculadora(expresion),
    "buscar_evento_universidad": lambda tema: buscar_evento_universidad(tema),
}


## 2. El loop del agente

In [ ]:
def run_agent(user_message: str, tools, implementations, max_steps: int = 5, verbose: bool = True):
    """Implementación manual del ciclo ReAct sobre la Interactions API de Gemini."""
    interaction = client.interactions.create(model=MODEL, input=user_message, tools=tools)

    for step_num in range(1, max_steps + 1):
        function_calls = [s for s in interaction.steps if s.type == "function_call"]

        # -- RAZONAR: si no hay function_call, el modelo ya dio su respuesta final --
        if not function_calls:
            if verbose:
                print(f"[Paso {step_num}] Respuesta final del agente.")
            return interaction.output_text

        # -- ACTUAR + OBSERVAR: ejecutamos cada function_call solicitado --
        function_results = []
        for fc in function_calls:
            if verbose:
                print(f"[Paso {step_num}] Actuando: {fc.name}({fc.arguments})")
            fn = implementations.get(fc.name)
            result = fn(**fc.arguments) if fn else f"Herramienta desconocida: {fc.name}"
            if verbose:
                print(f"[Paso {step_num}] Observando resultado: {result}")
            function_results.append({
                "type": "function_result",
                "name": fc.name,
                "call_id": fc.id,
                "result": [{"type": "text", "text": str(result)}],
            })

        interaction = client.interactions.create(
            model=MODEL, input=function_results, tools=tools,
            previous_interaction_id=interaction.id,
        )

    return "Se alcanzó el límite de pasos (max_steps) sin una respuesta final."


respuesta = run_agent(
    "Busca si hay algún evento de IA en la universidad, y si lo hay, dime cuántos días faltan si hoy es 18 de julio.",
    agent_tools, tool_implementations
)
print("\n=== RESPUESTA FINAL ===")
print(respuesta)


## 🧪 Ejercicio

Modifica `run_agent` para que imprima un **contador de tokens acumulado** en cada paso (usa `interaction.usage`, que expone los tokens consumidos). Esto es exactamente lo que herramientas de observabilidad como LangSmith o Langfuse hacen automáticamente — lo veremos en la Sesión 2.

---
**Guarda este notebook a mano** — la función `run_agent` la vamos a reutilizar tal cual en el notebook 6 (Taller 5).

**Siguiente notebook:** `langchain_agente.ipynb` — el mismo agente, reconstruido con un framework de orquestación.


In [ ]:
# Tu código aquí
